In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)


Usando dispositivo: cuda


In [2]:
import torch
from models.cnn.cnn_pure import CNNPure

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Cargar CNN entrenada
cnn_model = CNNPure(num_classes=2)
cnn_model.load_state_dict(
    torch.load("results/cnn/cnn_pure.pth", map_location=device)
)
cnn_model.to(device)
cnn_model.eval()

# Usar SOLO el extractor de características
feature_extractor = cnn_model.features

# Congelar pesos
for param in feature_extractor.parameters():
    param.requires_grad = False


In [3]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

IMG_SIZE = 384
BATCH_SIZE = 32
NUM_WORKERS = 2

# Transforms (usar los mismos que en la CNN)
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(
    root="dataset_split/train",
    transform=transform
)

val_dataset = datasets.ImageFolder(
    root="dataset_split/val",
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root="dataset_split/test",
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,  # 🔴 importante: NO mezclar para ELM
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print("Dataloaders listos")


Dataloaders listos


In [4]:
print("CNN cargada correctamente")

x, _ = next(iter(train_loader))
x = x.to(device)

with torch.no_grad():
    feats = cnn_model.features(x)

print("Shape de features:", feats.shape)


CNN cargada correctamente
Shape de features: torch.Size([32, 256])


In [5]:
import numpy as np

def extract_features(dataloader, model, device):
    features = []
    labels = []

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            feats = model.features(x)   # (batch, 256)
            features.append(feats.cpu().numpy())
            labels.append(y.numpy())

    X = np.vstack(features)
    y = np.concatenate(labels)
    return X, y


In [6]:
X_train, y_train = extract_features(train_loader, cnn_model, device)
X_val,   y_val   = extract_features(val_loader,   cnn_model, device)
X_test,  y_test  = extract_features(test_loader,  cnn_model, device)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


(560, 256) (560,)
(120, 256) (120,)


In [7]:
num_classes = len(np.unique(y_train))

def one_hot(y, num_classes):
    return np.eye(num_classes)[y]

Y_train_oh = one_hot(y_train, num_classes)
Y_val_oh   = one_hot(y_val, num_classes)


In [8]:
class ELM:
    def __init__(self, input_dim, hidden_dim, output_dim,
                 activation="relu", reg=1e-3):
        self.hidden_dim = hidden_dim
        self.reg = reg

        self.W = np.random.randn(input_dim, hidden_dim)
        self.b = np.random.randn(hidden_dim)

        if activation == "sigmoid":
            self.act = lambda x: 1 / (1 + np.exp(-x))
        elif activation == "tanh":
            self.act = np.tanh
        elif activation == "relu":
            self.act = lambda x: np.maximum(0, x)

    def fit(self, X, T):
        H = self.act(X @ self.W + self.b)
        I = np.eye(H.shape[1])
        self.beta = np.linalg.inv(H.T @ H + self.reg * I) @ H.T @ T

    def predict(self, X):
        H = self.act(X @ self.W + self.b)
        return np.argmax(H @ self.beta, axis=1)


In [46]:
elm = ELM(
    input_dim=256,
    hidden_dim=3600,   # puedes probar 500, 1000, 2000
    output_dim=num_classes,
    activation="tanh",
    reg=1e-3
)

elm.fit(X_train, Y_train_oh)
print("ELM entrenado")


ELM entrenado


In [47]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = elm.predict(X_test)

print("Accuracy CNN-ELM:", accuracy_score(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy CNN-ELM: 0.95

Classification report:
              precision    recall  f1-score   support

           0       0.94      0.97      0.95        60
           1       0.97      0.93      0.95        60

    accuracy                           0.95       120
   macro avg       0.95      0.95      0.95       120
weighted avg       0.95      0.95      0.95       120


Confusion matrix:
[[58  2]
 [ 4 56]]
